# 2024-11-04 Removing atmospheric effects, round 1

## Overview

From last time, you have measuresments of the brightness of the stars in your color image.

Today we will talk about how to remove the effects of the atmosphere and our instruments on those measurements.

<!-- ![Sketch of atmospheric and intrumental effects](media/AST-266-27.jpg) -->

In [ ]:
from pathlib import Path
import numpy as np

from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table

from astroquery.gaia import Gaia

from stellarphot import PhotometryData

%matplotlib inline
import matplotlib.pyplot as plt

#### 👇👇 put in the name of your file here 👇👇

In [ ]:
your_mag_table = 'photometry.ecsv'

In [ ]:
mag_table = PhotometryData.read(your_mag_table)

In [ ]:
B = mag_table[mag_table['passband'] == 'B']
V = mag_table[mag_table['passband'] == 'V']

In [ ]:
mag_col = 'mag_inst'
# fig, resid = plt.subplots(1, 1, figsize=(10, 10))

# resid.plot(B['color_cat'], B['color_cat'] - (B[mag_col] - V[mag_col]), '.', label='catalog colors', alpha=0.4)
# #plt.ylim(16.5, 10)
# resid.set_xlabel('Catalog B-V')
# resid.set_ylabel('Diff between catalog and calibrated B-V')
# resid.set_title(your_mag_table)
# resid.grid()

In [ ]:
good_color = np.abs(B[mag_col]) > -1  # B['color_cat'] - (B['mag_inst_cal'] - V['mag_inst_cal'])) < 0.1

In [ ]:
fig, cmd = plt.subplots(1, 1, figsize=(5, 5))

cmd.plot(B[mag_col][good_color] - V[mag_col][good_color], V[mag_col][good_color], '.')
cmd.set_ylim(*cmd.get_ylim()[::-1])
cmd.set_ylabel('V')
cmd.set_xlabel('B - V')
cmd.set_xlim(-0.5, 2.5)
cmd.set_xlabel("Color (B-V instrumental)")
cmd.set_ylabel("Magnitude (V instrumental")
cmd.grid()


## Atmospheric and instrumental effects

Discussion on the ipad....

<!-- ![Sketch of atmospheric and intrumental effects](media/AST-266-26.jpg) -->

## Calibrating your magnitudes

In [ ]:
from pathlib import Path
import numpy as np

from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table

from astroquery.gaia import Gaia

from stellarphot import PhotometryData
from ipyautoui.custom import FileChooser

from astropy.coordinates import SkyCoord
from stellarphot import apass_dr9, PhotometryData
%matplotlib widget
from matplotlib import pyplot as plt

import numpy as np

In [ ]:
chooser = FileChooser(filter_pattern=["*.ecsv"])
chooser

In [ ]:
YOUR_OBJECT_NAME = "M33"
coocoo = SkyCoord.from_name(YOUR_OBJECT_NAME)

In [ ]:
your_mag_table = chooser.value

In [ ]:
pd = PhotometryData.read(chooser.value)
pd = pd[pd["passband"] == "V"]

In [ ]:
dr9 = apass_dr9(coocoo)
dr9_good = dr9.passband_columns(passbands=["B", "V", "SR"])
dr9_coord = SkyCoord(dr9_good['ra'], dr9_good['dec'], unit='degree')

In [ ]:
our_coord = SkyCoord(pd['ra'], pd['dec'])
idx, d2d, _ = our_coord.match_to_catalog_sky(dr9_coord)
good_matches = d2d.arcsec < 1

In [ ]:
fig, ax = plt.subplots()
x = pd["mag_inst"][good_matches]
y = dr9_good[idx[good_matches]]["mag_V"].filled(np.nan)
b = dr9_good[idx[good_matches]]["mag_B"].filled(np.nan)
e = pd["mag_error"][good_matches].value

not_bad  = np.isfinite(x) & np.isfinite(y) & np.isfinite(e)
x = x[not_bad]
y = y[not_bad]
e = e[not_bad]
b = b[not_bad]

ax.errorbar(
    x, 
    y,
    xerr=e,
    fmt="."
)
ax.set_ylabel("Catalog V magnitude")
ax.set_xlabel("Feder instrumental V magnitude")
ax.grid()